In [20]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.neighbors import BallTree

# Unificação


In [4]:
data_dir = Path('../data/raw')

In [6]:
df_bosch = pd.read_excel(data_dir / 'bosch_service.xlsx')
df_bridgestone = pd.read_excel(data_dir / 'bridgestone.xlsx')
df_continental = pd.read_excel(data_dir / 'CONTINENTAL.xlsx')
df_michelin = pd.read_excel(data_dir / 'MICHELIN.xlsx')
df_pirelli = pd.read_excel(data_dir / 'pirelli.xlsx')
df_speedmax = pd.read_excel(data_dir / 'speedmax.xlsx')

In [9]:
mapa_bosch = {
    'Nome Comercial': 'nome', # Obs: Algumas lojas não têm Nome Comercial, trataremos abaixo
    'Cidade': 'cidade',
    'Estado': 'estado',
    'Latitude': 'lat',
    'Longitude': 'lon',
    'Telefone': 'telefone',
    'Email': 'email',
    'Website': 'website'
}

mapa_bridgestone = {
    'Nome': 'nome',
    'Cidade': 'cidade',
    'Estado': 'estado', # Traz a UF (ex: AC)
    'Latitude': 'lat',
    'Longitude': 'lon',
    'Telefone': 'telefone',
    'Email Gerente': 'email',
    'URL Página': 'website'
}

mapa_continental = {
    'Nome': 'nome',
    'Cidade': 'cidade',
    'Estado': 'estado',
    'Latitude': 'lat',
    'Longitude': 'lon',
    'Telefone': 'telefone',
    'Email': 'email',
    'Site': 'website'
}

mapa_michelin = {
    'Nome': 'nome',
    'Cidade': 'cidade',
    'Estado': 'estado',
    'Latitude': 'lat',
    'Longitude': 'lon',
    'Telefone': 'telefone'
    # Não possui email e website
}

mapa_pirelli = {
    'Nome': 'nome',
    'Cidade': 'cidade',
    'Estado': 'estado',
    'Latitude': 'lat',
    'Longitude': 'lon',
    'Telefone': 'telefone',
    'Email': 'email',
    'Website': 'website'
}

mapa_speedmax = {
    'Nome': 'nome',
    'Cidade': 'cidade',
    'Estado': 'estado',
    'Latitude': 'lat',
    'Longitude': 'lon',
    'Telefone': 'telefone'
    # Não possui email e website
}

In [10]:
df_bosch['Nome Comercial'] = df_bosch['Nome Comercial'].fillna(df_bosch['Razão Social'])

In [11]:
redes_setup = [
    (df_bosch, mapa_bosch, 'Bosch'),
    (df_bridgestone, mapa_bridgestone, 'Bridgestone'),
    (df_continental, mapa_continental, 'Continental'),
    (df_michelin, mapa_michelin, 'Michelin'),
    (df_pirelli, mapa_pirelli, 'Pirelli'),
    (df_speedmax, mapa_speedmax, 'Speedmax')
]

In [12]:
dfs_processados = []
colunas_finais = ['nome', 'rede', 'cidade', 'estado', 'lat', 'lon', 'telefone', 'email', 'website']

In [15]:
for df, mapa, nome_rede in redes_setup:
    # Renomeia as colunas
    df_temp = df.rename(columns=mapa).copy()
    
    # Adiciona a coluna da rede
    df_temp['rede'] = nome_rede
    
    # Garante que todas as colunas finais existam (preenche com NaN se faltar email/website)
    for col in colunas_finais:
        if col not in df_temp.columns:
            df_temp[col] = np.nan
            
    # Filtra apenas as colunas de interesse
    df_temp = df_temp[colunas_finais]
    dfs_processados.append(df_temp)

In [16]:
df_unificado = pd.concat(dfs_processados, ignore_index=True)

In [17]:
mask_lat = df_unificado['lat'].between(-34, 5)
mask_lon = df_unificado['lon'].between(-74, -28)

df_unificado = df_unificado[mask_lat & mask_lon]

In [18]:
print(f"Total de lojas unificadas (dentro do BR): {len(df_unificado)}")
print("\nTaxa de completude por campo (Geral):")
print((df_unificado.notnull().mean() * 100).round(2).astype(str) + '%')

df_unificado.head()

Total de lojas unificadas (dentro do BR): 6502

Taxa de completude por campo (Geral):
nome        100.0%
rede        100.0%
cidade      100.0%
estado      90.14%
lat         100.0%
lon         100.0%
telefone    79.22%
email       33.94%
website     41.29%
dtype: str


,nome,rede,cidade,estado,lat,lon,telefone,email,website
0,A. MORAIS CAMILO LTDA,Bosch,EPITACIOLANDIA,Acre,-11.257554,-68.740279,5568999251825.0,moraisandreia0210@gmail.com,NaN
1,Dalcar Auto Peças Ltda,Bosch,RIO BRANCO,Acre,-9.966927,-67.819428,556832244947.0,dalcarautopecas@hotmail.com,NaN
2,Varga Jatiúca,Bosch,MACEIO,Alagoas,-9.646906,-35.704858,5582988401027.0,financeirosm@vargamaceio.com.br,NaN
3,Varga Tabuleiro,Bosch,MACEIÓ,Alagoas,-9.595276,-35.750695,5582996757667.0,financeirotab@vargamaceio.com.br,NaN
4,ATALIBA MECÂNICA TÉCNICA,Bosch,MANAUS,Amazonas,-3.117503,-60.005192,559236633059.0,ataliba@atalibamecanica.com.br,NaN


# Deduplicação


In [21]:
coords = np.radians(df_unificado[['lat', 'lon']].values)

tree = BallTree(coords, metric='haversine')

raio_metros = 50
raio_terra_metros = 6371000
raio_radianos = raio_metros / raio_terra_metros

indices_vizinhos = tree.query_radius(coords, r=raio_radianos)

In [22]:
indices_para_manter = set()
indices_processados = set()

In [23]:
for vizinhos in indices_vizinhos:
    vizinhos_ordenados = sorted(vizinhos)
    loja_principal = vizinhos_ordenados[0] # Escolhe a primeira loja do cluster como a "oficial"
    
    if loja_principal not in indices_processados:
        indices_para_manter.add(loja_principal)
        # Marca todas as lojas deste raio como processadas para não duplicar
        for v in vizinhos_ordenados:
            indices_processados.add(v)

In [25]:
df_final = df_unificado.iloc[list(indices_para_manter)].reset_index(drop=True)
print(f"Total de lojas originais (pós-bbox): {len(df_unificado)}")
print(f"Total de lojas após remover duplicatas espaciais: {len(df_final)}")
print(f"Lojas duplicadas removidas: {len(df_unificado) - len(df_final)}")

Total de lojas originais (pós-bbox): 6502
Total de lojas após remover duplicatas espaciais: 3578
Lojas duplicadas removidas: 2924


In [26]:
caminho_saida = data_dir.parent / 'processed' / 'lojas_unificadas_limpas.csv'
df_final.to_csv(caminho_saida, index=False)
print(f"\nArquivo final salvo em: {caminho_saida}")


Arquivo final salvo em: ..\data\processed\lojas_unificadas_limpas.csv
